<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Crypto Foundations
## Chapter 1 – Why Cryptography Matters

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


In this first hands-on example you use Python's standard `hashlib` module to compute a SHA-256 digest of a message and then recompute it later to detect tampering. This mirrors the chapter's theme that integrity is about spotting unauthorized changes: if even a single bit flips, the digest comparison will fail and you know the message is no longer trustworthy.

In [1]:
import hashlib  # standard library hashing functions

message = b"pay alice 10"  # plaintext bytes representing a payment instruction

# Compute a SHA-256 digest for integrity checking
digest = hashlib.sha256(message).hexdigest()  # hex-encoded hash of the message
print("Digest:", digest)  # show the digest to compare later

# Later, recompute and compare to detect tampering
received = b"pay alice 10"  # try changing this to simulate an active adversary
check = hashlib.sha256(received).hexdigest()  # recompute digest of received bytes
print("Matches?", digest == check)  # True only if contents are unchanged


Digest: 37fd94ae6cdbaab54d81db99e742e7b1f19458186d364c464137d8670c83f77b
Matches? True


Next you tighten the guarantee from "nothing changed" to "someone with a shared secret endorsed this" by adding an HMAC tag. This code shows how a client and server that both know the key can independently compute the same authentication tag over a canonical request string and then use a constant-time comparison to decide whether to accept or reject the message.

In [2]:
import hmac  # keyed hashing for message authentication
import hashlib  # underlying hash function for HMAC

key = b"super-secret-key"      # shared only with trusted parties
msg = b"/v1/transfer|alice|bob|10"  # canonical request fields joined with separators

tag = hmac.new(key, msg, hashlib.sha256).hexdigest()  # compute HMAC over the message
print("Tag:", tag)  # display the authentication tag

# Verifier recomputes with the same key and message
check = hmac.new(key, msg, hashlib.sha256).hexdigest()  # recompute tag on the verifier side
print("Valid?", hmac.compare_digest(tag, check))  # constant-time comparison for safety


Tag: 78f90898d9590b011ba2f1ae070e72a91ff10a82bb20b809f119c33548742318
Valid? True


Finally, you capture the surrounding threat model directly in code so readers and reviewers can see what asset you are protecting, which goals matter, and which adversaries and assumptions you have in mind. Treat this as a lightweight, executable design note that lives next to your experiments and keeps cryptographic choices tied to explicit context.

In [3]:
threat_model = {  # minimal structured description of risks and goals
    "asset": "payment API",  # what we are protecting
    "goals": ["confidentiality", "integrity", "authenticity"],  # desired security properties
    "adversaries": [
        "network eavesdropper",  # passive listener on the wire
        "active MITM on public Wi-Fi",  # man-in-the-middle who can modify traffic
    ],
    "assumptions": [
        "server private keys stay on HSM",  # key material lives in hardware security module
        "clients validate TLS certificates",  # clients check server identity correctly
    ],
}

for k, v in threat_model.items():  # iterate over model entries for display
    print(f"{k.upper()}: {v}")  # print each key/value pair in a readable format


ASSET: payment API
GOALS: ['confidentiality', 'integrity', 'authenticity']
ADVERSARIES: ['network eavesdropper', 'active MITM on public Wi-Fi']
ASSUMPTIONS: ['server private keys stay on HSM', 'clients validate TLS certificates']


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
